<a href="https://colab.research.google.com/github/chiaranovelli/DII4ET_exercise_naples/blob/main/ROM_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reduced-Order Modeling: Hydrogen flashback tutorial

## Introduction

This notebook uses a dataset obtained from **numerical simulations of a hydrogen flame**.  
The data are stored in a **2D snapshot matrix** `X` with dimensions `(n_space, n_time)`, where

- `n_space = n_features × nx × ny`  
- `nx = 601`, `ny = 201`   
  - x ∈ [0, 0.06] m  
  - y ∈ [-0.01, 0.01] m  
- `n_features = 5`
  1. Density (ρ)  
  2. Mass fraction of H₂  
  3. Mass fraction of H₂O  
  4. Mass fraction of OH  
  5. Temperature (T)
- `n_time = 300`

This dataset represents the **temporal evolution of key thermochemical quantities** in a premixed flame, and will be used for:  

- loading and inspecting simulation data  
- performing **Proper Orthogonal Decomposition (POD)**  
- applying **linear regression model** on the reduced space.
- applying **Gaussian Process Regression (GPR)** on the reduced space.
- interpolation on unseen timestep



# Exercise 1: Preprocessing of the data

First we load the dataset from drive.

In [ ]:
!pip install -q gdown

In [ ]:
file_id = "1APMYWOGaGQEIHZIvm1VD7HI6qK5kYOnQ"
out_name = "X_300.npy"
!gdown --id {file_id} -O {out_name}

import numpy as np
X = np.load(out_name)
print("Loaded:", out_name, "Shape:", X.shape)

To examine the structure of the dataset, we first create a plot.

In [ ]:
# plotting function
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

nx = 601
ny = 201
x_coords = np.linspace(0, 0.06, nx)
y_coords = np.linspace(-0.01, 0.01, ny)
x, y = np.meshgrid(x_coords, y_coords, indexing='xy')
xy = np.column_stack([x.ravel(order='F'), y.ravel(order='F')])

features = ["rho","H2","H2O","OH","T"]
symbols_l = [r'$\rho$', r'$Y_{\mathrm{H}_2}$', r'$Y_{\mathrm{H}_2\mathrm{O}}$', r'$Y_{\mathrm{OH}}$', r'T (K)']
n_features = len(features)

n_space, n_time = X.shape

def plot_features(X, t , feature):
    feat_idx = features.index(feature)
    field = X[feat_idx*ny*nx:(feat_idx+1)*ny*nx, t].reshape((ny, nx))
    plt.figure(figsize=(7,2))
    plt.imshow(field, origin="lower", aspect="auto",
               extent=[x_coords[0], x_coords[-1], y_coords[0], y_coords[-1]],cmap='inferno')
    plt.colorbar(label=feature)
    plt.xlabel("x [m]"); plt.ylabel("y [m]")
    plt.title(f"{feature} at timestep {t}")
    plt.show()

plot_features(X, 200, 'T')

To gain a better understanding of the flashback phenomenon, we will create an animation:

In [ ]:
# animation creation
import matplotlib.animation as animation
from IPython.display import HTML

def animate_feature(X, feature, t_start=0, t_end=None, step=10, cmap="inferno",
                    dynamic_clim=False, p_low=1, p_high=99):
    """Animate one feature over time with proper color scaling.
       dynamic_clim=False -> global robust colorbar (default)
       dynamic_clim=True  -> colorbar adapts each frame
    """
    if feature not in features:
        raise ValueError(f"{feature} not in {features}")
    fidx = features.index(feature)
    start = fidx * ny * nx
    end   = (fidx + 1) * ny * nx

    n_time = X.shape[1]
    if t_end is None: t_end = n_time
    frame_list = list(range(t_start, min(t_end, n_time), step))

    fig, ax = plt.subplots(figsize=(7, 2))
    x_mg, y_mg = np.meshgrid(x_coords, y_coords)

    vmin = vmax = None
    if not dynamic_clim:
        stride = max(1, len(frame_list)//20)
        sample_idx = frame_list[::stride] or frame_list
        data_block = X[start:end][:, sample_idx]
        finite = np.isfinite(data_block)
        if finite.any():
            vmin = np.percentile(data_block[finite], p_low)
            vmax = np.percentile(data_block[finite], p_high)
        else:
            vmin, vmax = 0.0, 1.0
        del data_block

    field0 = X[start:end, frame_list[0]].reshape((ny, nx))
    im = ax.imshow(field0, origin="lower", aspect="auto",
                   extent=[x_coords[0], x_coords[-1], y_coords[0], y_coords[-1]],
                   cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, label=feature)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")

    def update(t):
        field = X[start:end, t].reshape((ny, nx))
        im.set_data(field)
        ax.set_title(f"{feature} at timestep {t}")
        if dynamic_clim:
            finite = np.isfinite(field)
            if finite.any():
                vmin_t = np.percentile(field[finite], p_low)
                vmax_t = np.percentile(field[finite], p_high)
                im.set_clim(vmin_t, vmax_t)
                cbar.update_normal(im)
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=frame_list, interval=100, blit=False)
    plt.close(fig)
    return ani


HTML(animate_feature(X, 'T', t_start=0, t_end=299, step=10).to_jshtml())

## Test and Training splitting
We have now to select some timesteps to save for testing the model later. So we split it in train and test randomly. Since we are gonna predict point inside the max range of our training set this is called interpolation. To predict outside of the training range is called extrapolation.

We need to select some timesteps to save for testing the model later.
So we split the dataset into train and test randomly.

Since we are going to predict points within the range of our training set,
this is called INTERPOLATION.
Predicting outside the training range is called EXTRAPOLATION.

<img src="https://towardsdatascience.com/wp-content/uploads/2024/07/1kFy08MaOKkjmgkphtZgmpw-1.png" width="600">



In [ ]:
# Exercise 1.
# split the datasetin test and training randomly. Choose the timestep ramdomly

# Hint:
# - use np.random.choice() to randomly select indices
# - np.sort() to order the indices
# - use np.setdiff1d() to find elements in one array but not in another

# we need train_idx, test_idx, X_train, X_test, t_train, t_test

K_train = 100  # n of train timestep
n_time = X.shape[1]
np.random.seed(42)

t = np.arange(n_time).reshape(-1,1)
t_norm = t / t.max()  # normalization of the time vector

# Randomly select training indices
train_idx = np.sort(np.random.choice(n_time, K_train, replace=False))

# Select test indices from remaining
test_idx = np.setdiff1d(np.arange(n_time), train_idx)

# Split snapshot matrix
X_train = X[:, train_idx]
X_test  = X[:, test_idx]

# Split time vector
t_train = t_norm[train_idx].reshape(-1,1)
t_test  = t_norm[test_idx].reshape(-1,1)

print(f"Training set: {X_train.shape} ({K_train} snapshots)")
print(f"Test set: {X_test.shape} ({n_time - K_train} snapshots)")

## Scaling and Centering
Some features (like the temperature) have a very different range of values respect to the other features like species that are limited between 0 and 1. Before applying methods like POD, it is important to normalize the dataset so that all physical quantities (e.g., temperature, species mass fractions) contribute comparably.

For each feature $ f $, we compute the mean and standard deviation:

\begin{align}
\mu_f = \text{mean}(X_f), \quad
\sigma_f = \text{std}(X_f)
\end{align}

and apply the normalization:

\begin{align}
X_{0f} = \frac{X_f - \mu_f}{\sigma_f}
\end{align}

This ensures that all features have mean = 0 and standard deviation = 1.

The function returns the scaled snapshots $ X_0$, the feature means $ X_{\text{cnt}} $,
and the standard deviations $ X_{\text{scl}} $.


In [ ]:
# scaling and unscaling functions

def feature_scale(X, n_features, ny, nx):
    """
    Per-feature standardization.
    X: snapshot matrix (n_space, n_time)

    Returns:
    X0: scaled snapshot matrix (n_space, n_time)
    X_cnt: array of feature means (n_features,)
    X_scl: array of feature standard deviations (n_features,)
    """
    X0 = np.zeros_like(X)
    X_cnt = []
    X_scl = []

    for f in range(n_features):
        Xi = X[f*ny*nx:(f+1)*ny*nx, :] #slicing for each features

        # calculate the mean and the std foe each features
        mu = Xi.mean()
        sigma = Xi.std()
        if sigma == 0:
            sigma = 1.0

        # scaling
        X0[f*ny*nx:(f+1)*ny*nx, :] = (Xi - mu) / sigma

        # save
        X_cnt.append(mu)
        X_scl.append(sigma)

    return X0, np.array(X_cnt), np.array(X_scl)

def reverse_feature_scale(X0, n_features, ny, nx, X_cnt, X_scl):
    """
    Reverse of per-feature standardization.

    Returns:
    X_rec in original units (n_space, n_time)
    """
    X_rec = np.zeros_like(X0)

    for f in range(n_features):
        X_rec[f*ny*nx:(f+1)*ny*nx, :] = X0[f*ny*nx:(f+1)*ny*nx, :] * X_scl[f] + X_cnt[f]

    return X_rec


We apply the scaling on the training dataset and use the same parameter for the test to be able to reconstruct after the reduction without bias.

In [ ]:
# Scale the training dataset
X0_train, X_cnt, X_scl = feature_scale(X_train, n_features, ny, nx)

# Scale the test dataset using the training stats only (to avoid bias)
X0_test  = np.zeros_like(X_test)
for f in range(n_features):
    X0_test[f*ny*nx:(f+1)*ny*nx, :] = (X_test[f*ny*nx:(f+1)*ny*nx, :]
                                               - X_cnt[f]) / X_scl[f]

In [ ]:
# --- Quick numeric check ---
print("Not scaled:")
print("X_train_mean:", np.mean(X_train))
print("X_train_cnt:", np.std(X_train))

print("Scaled:")
print("X0_train_mean:", np.mean(X0_train))
print("X0_train_cnt:", np.std(X0_train))


Now that we understand how the dataset is made let's apply dimensionality reduction to work with a smaller set of parameters. In particular we use Proper Orthogonal Decomposition - POD (also know as Principal Component Analysis - PCA).

# Exercise 2. Proper Orthogonal Decomposition

POD is dimentionality reduction technique used to

*   Identify the most energetic structures in the flow.
*   Compress the data.

There are 3 ways to compute the POD decomposition:

* Eigendecomposition of the spatial correlation matrix (classic POD).
* Eigendecomposition of the temporal correlation matrix (snapshot POD).
* Singular value decomposition.

Today we use **Eigendecomposition of the temporal correlation matrix**.

To apply the POD, the dataset has to be arranged as a data matrix $\mathbf{X} \in \mathbb{R}^{n \times m}$, where $n$ represents the number of points in space (pixels) and $m$ is the number of timesteps.

## POD algorithm

The first step of the snapshot POD algorithm consists in computing the temporal correlation matrix:

$$\mathbf{K} = \mathbf{X}_0^T \mathbf{X}_0$$

where each element $K_{i,j}$ is obtained as the inner product between the $i$-th and $j$-th snapshots: $K_{i,j} = \langle \mathbf{x}_i, \mathbf{x}_j \rangle$.

Note that $\mathbf{K} \in \mathbb{R}^{m \times m}$, which is much smaller than the spatial correlation matrix when $n >> m$.

Then, we compute the eigendecomposition of the temporal correlation matrix:

$$\mathbf{K} = \mathbf{V} \mathbf{L} \mathbf{V}^T$$

The eigenvectors $\mathbf{V}$ are the principal components in the temporal space, and the eigenvalues $\mathbf{L}$ represent the energy content of each mode.

The singular values are obtained as:

$$\mathbf{S} = \sqrt{\mathbf{L}}$$

The spatial POD modes are found by transforming back to the spatial domain:

$$\mathbf{U} = \mathbf{X}_0 \mathbf{V} \mathbf{S}^{-1}$$

The temporal coefficients are computed as:

$$\mathbf{A} = \mathbf{V} \mathbf{S}$$

This gives us the decomposition:

$$\mathbf{X}_0 = \mathbf{U} \mathbf{A}^T$$

which is equivalent to the truncated singular value decomposition.

## Properties

- The POD decouples the spatial information in $\mathbf{U}$ from the temporal information in $\mathbf{A}$.

- Both $\mathbf{U}$ and $\mathbf{V}$ are orthonormal.

- The eigenvalues (or singular values) represent the (relative) information content, and are thus used to identify the most important modes.

## Why Snapshot Method?

When $n >> m$ (many spatial points, few timesteps), computing the eigendecomposition of $\mathbf{K} \in \mathbb{R}^{m \times m}$ is much more efficient than computing the spatial correlation matrix $\mathbf{C} = \mathbf{X}_0\mathbf{X}_0^T \in \mathbb{R}^{n \times n}$.

## Truncation

The POD modes are ordered by decreasing energy content (eigenvalues). This means that the first few modes capture most of the variance in the data, while the later modes contain less important information.

We can exploit this property to achieve dimensionality reduction by keeping only the first $r$ modes and discarding the rest:

$$\mathbf{X}_0 \approx \mathbf{U}_r \mathbf{A}_r^T$$

where $\mathbf{U}_r \in \mathbb{R}^{n \times r}$ and $\mathbf{A}_r \in \mathbb{R}^{m \times r}$ contain only the first $r$ columns of $\mathbf{U}$ and $\mathbf{A}$.

The key question is: **how many modes should we keep?**

One way to choose $r$ is to select a threshold of the retained variance (or energy) equal to a high percentage of the total variance, such as 90%, 95%, or 98%.

The relative variance captured by each mode is:

$$V(i) = \frac{l_i}{\sum_{j=1}^{m} l_j}$$

where $l_i$ are the eigenvalues of $\mathbf{K}$.

The cumulative variance captured by the first $r$ modes is:

$$V(r) = \frac{\sum_{i=1}^{r} l_i}{\sum_{j=1}^{m} l_j}$$

To select $r$, we find the smallest number of modes such that:

$$V(r) \geq \text{threshold}$$

For example, if we want to retain 98% of the total energy, we choose $r$ such that $V(r) \geq 0.98$. This ensures that the reconstruction error remains small while significantly reducing the dimensionality of the problem ($r \ll m \ll n$).

In [ ]:
# Exercise 2:
# - Compute the eigendecomposition of the temporal correlation matrix
# - Plot the relative variance of the first 20 eigenvalues
# - Plot their cumulative relative energy

# Hints:
# - Matrices can be multiplied using the function np.dot() or @
# - To compute the eigendecomposition, use the function np.linalg.eig()
# - To plot, use the function plt.plot() or plt.scatter()

# We apply POD using Singular Value Decomposition (SVD): X = UΣV^T
# Instead of SVD on X (n_space × n_time), compute eigen-decomposition of the correlation matrix:
# C = X^T @ X ∈ R^(n_time ​× n_time​)
# V, S^2 = eig(C)
# Remember to sort V, S^2
# Recover spatial modes: U = XVΣ^−1

# Extra. Perform POD using direct SVD decomposition:
# More feasible, but we are not chosing it because of large number of columns, i.e., 900
# X = U Σ V^T → U: spatial modes, S: singular values, V^T: temporal structure
# Temporal coefficients: A = V Σ = (Σ V^T)^T

import time

def decompose(X):
    """
    POD using the snapshot method (efficient when n_space >> n_time).
    X: snapshot matrix (n_space, n_time)
    """
    # I. Snapshot method: eigendecomposition of temporal correlation
    C = X.T @ X                              # (n_time, n_time)
    eigvals, V = np.linalg.eigh(C)           # symmetric eigendecomposition

    # Sort eigenvalues descending
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    V = V[:, idx]

    # Singular values
    S = np.sqrt(np.maximum(eigvals, 0.0))

    # Spatial modes
    U = (X @ V) / S

    # Temporal coefficients
    A = V * S
    return U, A, S

# Compute POD
decompose_start_time = time.time()
U, A, S = decompose(X0_train)

decompose_end_time = time.time()
decompose_elapsed_time = decompose_end_time - decompose_start_time
print(f"\nTotal decomposition time: {decompose_elapsed_time:.2f} seconds")

Let's plot the relative energy for each mode and the cumulative energy:

In [ ]:
# POD Energy Plots
import numpy as np
import matplotlib.pyplot as plt

def plot_pod_energy(S, target=98.0, max_modes_to_plot=50):
    """
    Plot POD energy distribution and cumulative variance.

    Parameters
    ----------
    S : array-like
        Singular values from POD.
    target : float, optional
        Desired cumulative variance percentage (default = 98.0).
    max_modes : int, optional
        Maximum number of modes to display (default = 100).
    """
    # Energy and cumulative energy (%)
    energy = S**2
    total_energy = np.sum(energy)
    energy_pct = 100.0 * energy / total_energy
    cum_energy = 100.0 * np.cumsum(energy) / total_energy

    # Mode index reaching the target variance
    r_target = np.searchsorted(cum_energy, target) + 1
    print(f"→ {r_target} modes are needed to capture {target:.1f}% of the variance.")

    # Limit to the first `max_modes`
    n_show = min(max_modes_to_plot, len(energy))
    modes = np.arange(1, n_show + 1)

    # 1 - Energy per mode
    plt.figure(figsize=(8, 4))
    plt.bar(modes, energy_pct[:n_show], color='cornflowerblue', edgecolor='k')
    plt.xlabel("Mode index", fontsize=12)
    plt.ylabel("Energy per mode (%)", fontsize=12)
    plt.title(f"POD Energy per Mode ", fontsize=13)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # 2 - Cumulative energy with target line
    plt.figure(figsize=(8, 4))
    plt.plot(modes, cum_energy[:n_show], marker='o', color='darkorange', label='Cumulative energy')
    plt.axhline(y=target, color='red', linestyle='--', linewidth=1.5, label=f'{target:.0f}% threshold')


    # Vertical line if within plotted range
    if r_target <= n_show:
        plt.axvline(x=r_target, color='gray', linestyle='--', linewidth=1.2)
        plt.text(r_target + 2, target - 4, f'{r_target} modes', color='black', fontsize=11)

    plt.xlabel("Mode index", fontsize=12)
    plt.ylabel("Cumulative energy (%)", fontsize=12)
    plt.title(f"POD Cumulative Energy ", fontsize=13)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    return r_target

target=99
r = plot_pod_energy(S, target, max_modes_to_plot=20)

print(f'to reach {target}% of the variance we need {r} modes')



We don't need all the 500 modes, with 16 modes we capture 98\% of the variance. So to reduce the dimensionality of the problem we cut the POD modes and coefficient accordilly. Next, we normalize the temporal coefficients Ar to obtain an orthonormal temporal basis Vr, where each column has unit norm. This step ensures that all temporal modes are scaled consistently, making them more suitable for regression or machine learning tasks (e.g., GPR, neural networks).

In [ ]:
def reduce(U, A, S, r):
    return U[:, :r], A[:, :r], S[:r]

Ur_train, Ar_train, Sr_train = reduce(U, A, S, r)

# Vr holds normalized versions of the columns of Ar
Vr_train = Ar_train / Sr_train

print(f"Number of POD modes kept: {Ur_train.shape[1]}")
print(f"Number of time steps: {Ar_train.shape[0]}")

Let's see the behaviour of the POD coefficients (Vr) in times

In [ ]:
# Plot POD coefficient in time

def plot_pod_time_coeffs(Ar, num_modes_to_plot, t_indices=None):
    """
    Plot temporal evolution of selected POD modes.
    Ar: time coefficients (n_time, r)
    t_indices: actual time step indices (optional)
    """
    n_time, r = Ar.shape
    num_modes_to_plot = min(num_modes_to_plot, r)

    # Use provided time indices or default to sequential
    x_axis = t_indices if t_indices is not None else np.arange(n_time)

    fig, ax = plt.subplots(figsize=(7, 4.5))

    for i in range(num_modes_to_plot):
        ax.plot(x_axis, Ar[:, i], 'o-', label=f"Mode {i+1}", markersize=2)

    ax.set_xlabel("Time step")
    ax.set_ylabel("Coefficient value")
    ax.set_title(f"Temporal evolution of first {num_modes_to_plot} POD modes")
    ax.legend()
    ax.grid(True)
    plt.show()

plot_pod_time_coeffs(Vr_train, num_modes_to_plot=5,t_indices=train_idx)

# Exercise 3. Linear Regression

Another important feature of POD is that it allows us to easily build ROMs of combustion systems. A ROM exploits data compression to reduce the complexity of the full-order system and to build a regression model that can instantly predict the system's state.

## Linear Regression

**Linear Regression (LR)** assumes that the temporal evolution of each POD coefficient can be approximated by a linear function of time $ y = f(t)$:

$$\mathbf{v}_i(t) = \beta_{0,i} + \beta_{1,i} \cdot t$$

where $\mathbf{v}_i(t)$ is the normalized temporal coefficient for mode $i$, and $\beta_{0,i}$, $\beta_{1,i}$ are the regression parameters learned from the training data.

<img src="https://www.statisticalaid.com/wp-content/uploads/2025/05/Linear-Regression.png" width="400">




## Training

For each POD mode $i = 1, \ldots, r$, we fit a separate linear model:

$$\min_{\beta_{0,i}, \beta_{1,i}} \sum_{j \in \text{train}} \left( v_{i,j} - (\beta_{0,i} + \beta_{1,i} \cdot t_j) \right)^2$$

where $v_{i,j}$ is the normalized coefficient of mode $i$ at training timestep $j$.

## Prediction

Given a test time $t^*$, the predicted normalized coefficient is:

$$\hat{v}_i(t^*) = \beta_{0,i} + \beta_{1,i} \cdot t^*$$

## Limitations

Linear regression assumes a **linear relationship** between time and the POD coefficients. This is a strong assumption that is unlikely to hold for complex dynamical systems exhibiting nonlinear behaviors such as:
- Oscillations
- Periodic patterns
- Sudden transitions (e.g., flame ignition/extinction)

Therefore, we expect Linear Regression to perform **worse** than non linear regression, which can capture more complex temporal dynamics but it's always a good place to start.

Wea re gonna use the lybrary from scikit:
[LinearRegression Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [ ]:
# Exercise 3: Linear Regression Training & Prediction

# - Train a linear mode for each mode and predict the POD coefficient for the test timesteps.

# Hints:
# - Use LinearRegression from sklearn.linear_model
# - For each mode:
#   * Create a LinearRegression model
#   * Fit the model using t_train as input and Vr_train as target
#   * Predict on t_test and store results in Vr_pred_lr

from sklearn.linear_model import LinearRegression

Vr_pred_lr = np.zeros((len(t_test), r))

for i in range(r):
    # Train linear regression for each POD mode
    model_lr = LinearRegression()
    model_lr.fit(t_train, Vr_train[:, i])

    # Predict on test times
    Vr_pred_lr[:, i] = model_lr.predict(t_test)

# Exercise 4. Gaussian Process Regression

When the linear regression doesn't work, normal for complex combustion system there are multiple non linear regression that can be use like neural networks.
In this case we are gonna use Gaussian Process Regression (GPR), which is a Bayesian method that let us estimate also the uncertainty in the estimation.

## Gaussian Process Regression

A Gaussian Process defines a **distribution over functions**:

$$
v_i(t) \sim \mathrm{GP}\,\!\big(m(t),\, k(t,t')\big)
$$

where:
- \( m(t) \) is the mean function (often set to zero),
- \( k(t, t') \) is the covariance or **kernel function** that defines the smoothness and correlation structure of the data.

The idea is that any finite collection of function values follows a normal distribution:

$$
[v_i(t_1),\, \dots,\, v_i(t_n)]^{\top} \sim N\!\left(\mathbf{0},\, K\right)
$$

where \( $K$ \) is the covariance matrix computed from the kernel.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_gpr_noisy_targets_002.png" width="400">

## Training

For each POD mode \( $ i = 1, \dots, r $ \), GPR learns the **hyperparameters** of the kernel by maximizing the **log marginal likelihood** of the training data:

$$
\log p(\mathbf{v}_i | \mathbf{t}, \theta) = -\frac{1}{2}\mathbf{v}_i^\top K_\theta^{-1}\mathbf{v}_i - \frac{1}{2}\log |K_\theta| - \frac{n}{2}\log(2\pi)
$$

This allows GPR to automatically balance **model complexity** and **data fit** through its kernel structure.



## Prediction

Given a new time \( $t^*$ \), GPR predicts the mean and variance of the normalized coefficient:

$$
\hat{v}_i(t^*) = k_*^\top K^{-1}\mathbf{v}_i
$$

$$
\mathrm{Var}[\hat{v}_i(t^*)] = k(t^*,t^*) - k_*^\top K^{-1}k_*
$$

Thus, GPR not only provides a smooth nonlinear prediction but also a **confidence interval** around it.

### Implementation

We are going to use the GaussianProcessRegressor class from **scikit-learn**:

[GaussianProcessRegressor Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html)


In [ ]:
# Exercise 4: GPR Training & Prediction

# Train a separate GPR model for each POD mode to predict normalized coefficients at test times

# Hints:
# - For each mode:
#   * Define a kernel: use ConstantKernel * RBF to capture amplitude and smoothness,
#   * Adjust length_scale_bounds (length_scale=1.0, length_scale_bounds=(1e-3, 1e3)) for each kernel when necessary
#   * Create a GaussianProcessRegressor with the kernel
#   * Fit and predict

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

Vr_pred_gpr = np.zeros((len(t_test), r))  # Storage for predicted normalized coefficients
training_time_start = time.time()

for i in range(r):
    print(f"Training mode {i+1}/{r}")

    # Define kernel: Constant * RBF captures both amplitude and smoothness
    kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))

    gpr = GaussianProcessRegressor(kernel=kernel)

    # Fit: time normalized coefficient for mode i
    gpr.fit(t_train, Vr_train[:, i])

    # Predict normalized coefficients at test times
    Vr_pred_gpr[:, i] = gpr.predict(t_test)

training_time_end = time.time()
training_elapsed_time = training_time_end - training_time_start
print(f"\nTotal time: {training_elapsed_time:.2f} seconds")

# Exercise 5. Reconstruction and prediction error

Now we have the prediction of the linear and non linear model for this problem in the low dimensional space. Let's go back to the physical space and compare the results.


In [ ]:
# Run this cell to free the RAM of not needed variable, keep al the heavy matrix can be a problem

import gc
del X0_train # Elimina variabili intermediate
gc.collect()

In [ ]:
# Exercise 5. Reconstruction

# Reconstruct from the low dimensional space of the prediction Vr to the physical high dimensional space (T, rho, species).

# Hints:
# there are 3 reconstruction to do:
#  * from Vr to Ar (normalization of the POD coefficient)
#  * from Ar to X0 (POD)
#  * from X0 to X  (centering and scaling)


def reconstruction(Vr_pred):

  # Reverse the normalization: multiply by saved norms
  Ar_pred= Vr_pred * Sr_train  # (n_test, r)

  # Reconstruct test snapshots in scaled physical space
  X0_pred = Ur_train @ Ar_pred.T  # (n_space, n_test)

  # Transform predictions back to original physical units
  # using the mean and std
  X_pred = reverse_feature_scale(X0_pred, n_features, ny, nx, X_cnt, X_scl)

  del X0_pred # Elimina variabili intermediate
  gc.collect()

  return X_pred

X_pred_lr = reconstruction(Vr_pred_lr)
X_pred_gpr = reconstruction(Vr_pred_gpr)

Let's plot some timestep to see the recostruction of the different model respect to the original matrix

In [ ]:
# Visualize dynamics

def plot_flame_comparison(X_true, X_pred, timestep_idx, feature, cmap="inferno", levels=50, label=None, model_name = None):
    """
    Compare true vs predicted field for a given feature at a test timestep.
    Top = observed (X_true), Bottom = GPR predicted (X_pred).
    """
    if feature not in features:
        raise ValueError(f"Feature '{feature}' not in {features}")

    f = features.index(feature)

    field_true = X_true[f*ny*nx:(f+1)*ny*nx, timestep_idx].reshape((ny, nx))
    field_pred = X_pred[f*ny*nx:(f+1)*ny*nx, timestep_idx].reshape((ny, nx))

    x_mg, y_mg = np.meshgrid(x_coords, y_coords)

    combined = np.vstack((field_true[:ny//2, :], field_pred[ny//2:, :]))
    combined = np.flipud(combined)

    fig, ax = plt.subplots(figsize=(6, 2.5))
    cp = ax.contourf(x_mg, y_mg, combined, levels=levels, cmap=cmap)

    reduced_ticks = np.linspace(combined.min(), combined.max(), 10)

    # Move colorbar to right
    cbar_ax = fig.add_axes([0.955, 0.12, 0.01, 0.76])  # [left, bottom, width, height]
    cbar = fig.colorbar(cp, cax=cbar_ax, ticks=reduced_ticks)
    cbar.set_label(label if label else feature)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_yticks(np.linspace(y_coords[0], y_coords[-1], 5))

    # Labels outside plot
    x_label_pos = x_coords[-1] + 3e-4
    ax.text(x_label_pos, y_coords[3*ny//4], 'Obs.', va='center', ha='left', fontsize=10)
    ax.text(x_label_pos, y_coords[ny//4], model_name, va='center', ha='left', fontsize=10)

    ax.set_title(f"Obs. and Pred. {label} / t_idx = {timestep_idx}")
    plt.show()

symbols_l = [r'$\rho$', r'$Y_{\mathrm{H}_2}$', r'$Y_{\mathrm{H}_2\mathrm{O}}$',
             r'$Y_{\mathrm{OH}}$', r'T(K)']

feature = 'T'
f = features.index(feature)

plot_flame_comparison(X_test, X_pred_gpr, timestep_idx=89, feature=feature, label=symbols_l[f], model_name = 'gpr')
plot_flame_comparison(X_test, X_pred_lr, timestep_idx=89, feature=feature, label=symbols_l[f], model_name = 'lr')

In [ ]:
# Functions to compute and plot the errors

from sklearn.metrics import root_mean_squared_error
import matplotlib as mpl

def compute_feature_errors(X_test, X_rec, X_pred, features, nx, ny):
    """
    Compute NRMSE per feature for reconstruction and prediction.

    Parameters
    ----------
    X_test: np.ndarray: Ground truth test snapshots (n_space, n_time_test)
    X_rec: np.ndarray: POD reconstruction of X_test (same shape)
    X_pred: np.ndarray: GPR predicted snapshots (same shape)
    features : list of str
    nx, ny : int: Spatial resolution

    Returns
    -------
    rec_errors: dict: {feature: NRMSE} for reconstruction
    pred_errors: dict: {feature: NRMSE} for prediction
    """
    rec_errors, pred_errors = {}, {}

    for f, feat in enumerate(features):

        test = X_test[f*ny*nx:(f+1)*ny*nx, :].ravel()
        rec  = X_rec[f*ny*nx:(f+1)*ny*nx, :].ravel()
        pred = X_pred[f*ny*nx:(f+1)*ny*nx, :].ravel()

        rec_errors[feat]  = root_mean_squared_error(test, rec) / np.mean(test)
        pred_errors[feat] = root_mean_squared_error(test, pred) / np.mean(test)

    return rec_errors, pred_errors

def plot_error(error, features, labels, markers, colors, lims=None, loc='best'):
    """
    Scatter plot of per-feature errors.
    error: np.array of shape (n_models, n_features)
    """
    fig, ax = plt.subplots(figsize=(4.5, 3))

    num_models = error.shape[0]
    num_features = len(features)
    mean_errors = error

    shifts = np.linspace(-0.17, 0.17, num_models)
    for i in range(num_models):
        ax.scatter(np.arange(num_features) + shifts[i], mean_errors[i],
                   marker=markers[i], color=colors[i], label=labels[i], s=40, alpha=0.7)

    # Formatting
    ax.set_xticks(np.arange(num_features))
    ax.set_xticklabels(features, rotation=0)
    ax.set_ylabel("NRMSE")
    ax.set_yscale("log")
    ax.set_title("Errors")

    if lims is not None:
        ax.set_ylim(lims[0], lims[1])

    ax.legend(loc=loc, frameon=True, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
#%%
# Test data reconstruction error:
# To assess how much truncation alone costs (the lower bound of achievable prediction error),
# the test snapshots are projected onto the training POD basis and reconstruct.

# POD reconstruction error test
# Project test data onto POD modes to get test coefficients
Ar_rec_test = (Ur_train.T @ X0_test).T  # (n_test, r)

# True POD truncation error (from_train_basis)
X0_rec_test = Ur_train @ Ar_rec_test.T
X_rec_test = reverse_feature_scale(X0_rec_test, n_features, ny, nx, X_cnt, X_scl)

del X0_rec_test # Cancel some variables to save memory
gc.collect()

#%%
# Compute errors:
rec_errors, pred_errors = compute_feature_errors(X_test, X_rec_test, X_pred_gpr, features, nx, ny)
_, pred_errors_lr = compute_feature_errors(X_test, X_rec_test, X_pred_lr, features, nx, ny)

error_array = np.array([
    [rec_errors[f] for f in features],
    [pred_errors[f] for f in features],
    [pred_errors_lr[f] for f in features]
])

features_l = [r'$\rho$', r'$\mathrm{H}_2$', r'$\mathrm{H}_2\mathrm{O}$',
             r'$\mathrm{OH}$', r'T(K)']

labels = ["Reconst.(POD)", "Pred. (GPR)", "Pred. (LR)"]

markers = ['o', 's', '^']
cmap = mpl.colormaps['seismic']
colors = [cmap(i/2.9) for i in range(3)]

plot_error(error=error_array, features=features_l, labels=labels, markers=markers,
           colors=colors, lims=(1e-2, 1e1), loc='upper left')




# Extra Exercises

To practice more with the topic, we propose 2 extra exercises:

## 1. GPyTorch

Instead of the scikit-learn library, it's good to know **GPyTorch**. It is a powerful library that uses PyTorch and can handle complex machine learning problems. Try to repeat the exercise using GPyTorch.

[GPyTorch Documentation](https://docs.gpytorch.ai/en/stable/)

**Hints:**
- GPyTorch uses PyTorch tensors instead of NumPy arrays
- You need to define a GP model class inheriting from `gpytorch.models.ExactGP`
- Training uses gradient descent optimization (e.g., Adam optimizer)
- GPyTorch can be faster and more flexible for large-scale problems

## 2. Extrapolation

One of the limitations of GPR and non-physics-based regression in general is that they have much higher error for **extrapolation**. In this case, it means predicting future (or past) timesteps that are **outside** the training range.

As an exercise, try to extrapolate (predict in the future) and see how the prediction and the error change.

**Hints:**
- Modify the train/test split to use the **last K timesteps** as test set
- Compare the extrapolation error with the interpolation error from the main exercise
- Observe how GPR predictions often regress to the mean when extrapolating


In [ ]:
!pip install -q gpytorch

In [ ]:
# Extra exercise 1.

# Try to reproduce the GPR exercise using gpytorch library instead of scikit-learn. It's a more complex but more customizable model that can andle more complex problem.



In [ ]:
# Exercise extra 2. Extrapolations

# Try to do predictions in the future.
